In [ ]:

#  This notebook demonstrates:
#  - Reading secrets from Azure Key Vault via Databricks Secret Scope
#  - Connecting to Azure Data Lake Gen2
#  - Performing data analysis and transformation with PySpark
#  - Converting timestamps to datetime
#  - Flattening JSON columns
#  - Loading processed data back to Data Lake Gen2

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Import Required Libraries and Setup

# COMMAND ----------
# Importing libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
from datetime import datetime

# Initialize Spark session with optimized configurations
spark = SparkSession.builder \
    .appName("DataLakeGen2Processing") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

print(" Spark session initialized successfully")
print(f" Spark version: {spark.version}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Read Secrets from Azure Key Vault Secret Scope
# MAGIC 
# MAGIC Prerequisites:
# MAGIC - Create a secret scope linked to Azure Key Vault
# MAGIC - Store storage account credentials in Key Vault

# COMMAND ----------

# Define secret scope name (created in Databricks UI or CLI)
SECRET_SCOPE = "databricks-secrets"

# Read secrets from Azure Key Vault via Databricks Secret Scope
try:
    # Storage account credentials
    storage_account_name = dbutils.secrets.get(scope=SECRET_SCOPE, key="storage-account-name")
    storage_account_key = dbutils.secrets.get(scope=SECRET_SCOPE, key="storage-account-key")
    
    # Alternative: Use Service Principal (recommended for production)
    # client_id = dbutils.secrets.get(scope=SECRET_SCOPE, key="service-principal-client-id")
    # client_secret = dbutils.secrets.get(scope=SECRET_SCOPE, key="service-principal-client-secret")
    # tenant_id = dbutils.secrets.get(scope=SECRET_SCOPE, key="tenant-id")
    
    print(" Successfully retrieved secrets from Key Vault")
    print(f" Storage Account: {storage_account_name}")
    
except Exception as e:
    print(f" Error retrieving secrets: {str(e)}")
    print(" Make sure you've created a secret scope and stored the required secrets")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Configure Data Lake Gen2 Connection

# COMMAND ----------

# Configure Spark to connect to Data Lake Gen2
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

# Alternative configuration using Service Principal (recommended for production)
# spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
# spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
# spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
# spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
# spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Define Data Lake paths
source_container = "raw-data"
target_container = "processed-data"
source_path = f"abfss://{source_container}@{storage_account_name}.dfs.core.windows.net/"
target_path = f"abfss://{target_container}@{storage_account_name}.dfs.core.windows.net/"

print(" Data Lake Gen2 configuration completed")
print(f" Source Path: {source_path}")
print(f" Target Path: {target_path}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Read Data from Data Lake Gen2

# COMMAND ----------

# Read sample data from Data Lake Gen2
# Assuming we have employee data with labour_detail as JSON column
try:
    # Read data from parquet files (adjust path as needed)
    df_raw = spark.read \
        .option("inferSchema", "true") \
        .option("header", "true") \
        .parquet(f"{source_path}employee_data/")
    
    print(" Successfully read data from Data Lake Gen2")
    print(f" Total records: {df_raw.count()}")
    print(f" Schema:")
    df_raw.printSchema()
    
except Exception as e:
    print(f"❌ Error reading data: {str(e)}")
    print("💡 Creating sample data for demonstration...")
    
    # Create sample data for demonstration
    sample_data = [
        (1, "John Doe", "2024-06-04 14:30:25", '{"hourly_rate": 25.50, "overtime_hours": 8, "department": "Engineering", "skills": ["Python", "Spark", "Azure"]}'),
        (2, "Jane Smith", "2024-06-04 15:45:30", '{"hourly_rate": 28.75, "overtime_hours": 5, "department": "Data Science", "skills": ["SQL", "Python", "Machine Learning"]}'),
        (3, "Mike Johnson", "2024-06-04 16:20:10", '{"hourly_rate": 22.00, "overtime_hours": 12, "department": "Analytics", "skills": ["Tableau", "Power BI", "SQL"]}'),
        (4, "Sarah Wilson", "2024-06-04 17:15:45", '{"hourly_rate": 30.25, "overtime_hours": 3, "department": "Engineering", "skills": ["Java", "Kubernetes", "Docker"]}'),
        (5, "David Brown", "2024-06-04 18:00:20", '{"hourly_rate": 26.80, "overtime_hours": 10, "department": "DevOps", "skills": ["AWS", "Terraform", "Jenkins"]}')
    ]
    
    schema = StructType([
        StructField("employee_id", IntegerType(), True),
        StructField("employee_name", StringType(), True),
        StructField("timestamp_str", StringType(), True),
        StructField("labour_detail", StringType(), True)
    ])
    
    df_raw = spark.createDataFrame(sample_data, schema)
    print(" Sample data created successfully")

# Display sample data
print("\n Sample Data:")
df_raw.show(5, truncate=False)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Data Analysis and Transformation

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.1 Convert Timestamp String to DateTime

# COMMAND ----------

# Convert timestamp string to proper datetime format
df_with_datetime = df_raw.withColumn(
    "processed_timestamp", 
    to_timestamp(col("timestamp_str"), "yyyy-MM-dd HH:mm:ss")
)

# Add additional datetime columns for analysis
df_with_datetime = df_with_datetime \
    .withColumn("year", year(col("processed_timestamp"))) \
    .withColumn("month", month(col("processed_timestamp"))) \
    .withColumn("day", dayofmonth(col("processed_timestamp"))) \
    .withColumn("hour", hour(col("processed_timestamp"))) \
    .withColumn("day_of_week", dayofweek(col("processed_timestamp"))) \
    .withColumn("is_weekend", 
                when(dayofweek(col("processed_timestamp")).isin(1, 7), True).otherwise(False))

print(" Timestamp conversion completed")
print("\n Data with DateTime columns:")
df_with_datetime.select("employee_name", "timestamp_str", "processed_timestamp", "year", "month", "day", "hour", "is_weekend").show()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.2 Flatten JSON Column (labour_detail)

# COMMAND ----------

# Define schema for the JSON column
labour_detail_schema = StructType([
    StructField("hourly_rate", DoubleType(), True),
    StructField("overtime_hours", IntegerType(), True),
    StructField("department", StringType(), True),
    StructField("skills", ArrayType(StringType()), True)
])

# Parse JSON column and flatten it
df_flattened = df_with_datetime \
    .withColumn("labour_detail_parsed", from_json(col("labour_detail"), labour_detail_schema)) \
    .select("*", "labour_detail_parsed.*") \
    .drop("labour_detail", "labour_detail_parsed")

print(" JSON column flattened successfully")
print("\n Flattened Data Schema:")
df_flattened.printSchema()

print("\n Sample Flattened Data:")
df_flattened.show(truncate=False)

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.3 Advanced Data Analysis

# COMMAND ----------

# Perform additional transformations and analysis
df_analyzed = df_flattened \
    .withColumn("total_compensation", 
                col("hourly_rate") * (40 + col("overtime_hours") * 1.5)) \
    .withColumn("overtime_pay", 
                col("hourly_rate") * col("overtime_hours") * 1.5) \
    .withColumn("regular_pay", 
                col("hourly_rate") * 40) \
    .withColumn("skills_count", 
                size(col("skills"))) \
    .withColumn("is_high_performer", 
                when(col("overtime_hours") > 8, True).otherwise(False)) \
    .withColumn("experience_level",
                when(col("hourly_rate") >= 30, "Senior")
                .when(col("hourly_rate") >= 25, "Mid")
                .otherwise("Junior"))

print(" Advanced analysis completed")
print("\n Compensation Analysis:")
df_analyzed.select("employee_name", "hourly_rate", "overtime_hours", "total_compensation", "experience_level").show()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.4 Data Quality Checks and Validation

# COMMAND ----------

# Perform data quality checks
print(" Data Quality Report:")
print("=" * 50)

# Check for null values
null_counts = df_analyzed.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_analyzed.columns])
print("Null Value Counts:")
null_counts.show()

# Check data types
print("\nData Types:")
for field in df_analyzed.schema.fields:
    print(f"  {field.name}: {field.dataType}")

# Statistical summary
print("\nStatistical Summary:")
df_analyzed.select("hourly_rate", "overtime_hours", "total_compensation", "skills_count").describe().show()

# Department analysis
print("\nDepartment Analysis:")
df_analyzed.groupBy("department") \
    .agg(
        count("*").alias("employee_count"),
        avg("hourly_rate").alias("avg_hourly_rate"),
        avg("overtime_hours").alias("avg_overtime_hours"),
        avg("total_compensation").alias("avg_total_compensation")
    ).show()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5.5 Create Additional Derived Datasets

# COMMAND ----------

# Create aggregated datasets for different business needs

# 1. Employee Summary Dataset
df_employee_summary = df_analyzed.select(
    "employee_id",
    "employee_name",
    "department",
    "experience_level",
    "hourly_rate",
    "overtime_hours",
    "total_compensation",
    "skills_count",
    "is_high_performer",
    "processed_timestamp"
)

# 2. Department Summary Dataset
df_department_summary = df_analyzed.groupBy("department") \
    .agg(
        count("employee_id").alias("total_employees"),
        avg("hourly_rate").alias("avg_hourly_rate"),
        sum("overtime_hours").alias("total_overtime_hours"),
        avg("total_compensation").alias("avg_compensation"),
        countDistinct("experience_level").alias("experience_levels"),
        collect_set("skills").alias("all_skills")
    )

# 3. Skills Analysis Dataset
df_skills_analysis = df_analyzed.select("employee_id", "employee_name", "department", explode("skills").alias("skill")) \
    .groupBy("skill") \
    .agg(
        count("employee_id").alias("skill_count"),
        collect_list("department").alias("departments"),
        avg("hourly_rate").alias("avg_hourly_rate_for_skill")
    )

print(" Derived datasets created successfully")
print(f" Employee Summary: {df_employee_summary.count()} records")
print(f" Department Summary: {df_department_summary.count()} records")
print(f" Skills Analysis: {df_skills_analysis.count()} records")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Load Data Back to Data Lake Gen2

# COMMAND ----------

# MAGIC %md
# MAGIC ### 6.1 Write Processed Data to New Container

# COMMAND ----------

# Write main processed dataset
try:
    # Write as Parquet with partitioning for better performance
    df_analyzed.write \
        .mode("overwrite") \
        .partitionBy("department", "year", "month") \
        .option("path", f"{target_path}employee_processed/") \
        .saveAsTable("employee_processed")
    
    print(" Main processed dataset saved successfully")
    
    # Write employee summary
    df_employee_summary.write \
        .mode("overwrite") \
        .parquet(f"{target_path}employee_summary/")
    
    print(" Employee summary dataset saved successfully")
    
    # Write department summary
    df_department_summary.write \
        .mode("overwrite") \
        .parquet(f"{target_path}department_summary/")
    
    print(" Department summary dataset saved successfully")
    
    # Write skills analysis
    df_skills_analysis.write \
        .mode("overwrite") \
        .parquet(f"{target_path}skills_analysis/")
    
    print(" Skills analysis dataset saved successfully")
    
except Exception as e:
    print(f" Error writing data: {str(e)}")
    print(" Check container permissions and path configurations")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 6.2 Write Data in Different Formats

# COMMAND ----------

# Write data in different formats for various use cases

# 1. CSV format for business users
df_employee_summary.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{target_path}employee_summary_csv/")

# 2. JSON format for API consumption
df_department_summary.coalesce(1).write \
    .mode("overwrite") \
    .json(f"{target_path}department_summary_json/")

# 3. Delta format for data lakehouse (if Delta is available)
try:
    df_analyzed.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{target_path}employee_delta/")
    print(" Delta format saved successfully")
except:
    print("ℹ Delta format not available, skipping...")

print(" Multiple format exports completed")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 6.3 Create External Tables for SQL Access

# COMMAND ----------

# Create external tables for SQL access
try:
    # Create external table for processed data
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS processed_employee_data
        USING PARQUET
        LOCATION '{target_path}employee_processed/'
    """)
    
    # Create external table for department summary
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS department_summary_table
        USING PARQUET
        LOCATION '{target_path}department_summary/'
    """)
    
    print(" External tables created successfully")
    
    # Show available tables
    print("\n📋 Available Tables:")
    spark.sql("SHOW TABLES").show()
    
except Exception as e:
    print(f" Error creating tables: {str(e)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Data Validation and Quality Checks

# COMMAND ----------

# Validate written data by reading it back
print("🔍 Validating written data...")

try:
    # Read back the processed data
    df_validation = spark.read.parquet(f"{target_path}employee_processed/")
    
    print(f" Validation successful - {df_validation.count()} records found")
    print(" Sample of written data:")
    df_validation.select("employee_name", "department", "total_compensation", "experience_level").show(5)
    
    # Compare record counts
    original_count = df_analyzed.count()
    written_count = df_validation.count()
    
    if original_count == written_count:
        print(f" Record count validation passed: {original_count} = {written_count}")
    else:
        print(f" Record count mismatch: Original={original_count}, Written={written_count}")
        
except Exception as e:
    print(f" Validation error: {str(e)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Performance Optimization and Cleanup

# COMMAND ----------

# Optimize and cleanup
print(" Performing cleanup and optimization...")

# Cache frequently used DataFrames
# df_analyzed.cache()

# Unpersist cached DataFrames to free memory
# df_analyzed.unpersist()

# Clear catalog cache
# spark.catalog.clearCache()

# Show execution plan for complex operations (optional)
# print(" Execution Plan for Complex Transformation:")
# df_analyzed.explain(True)

# Collect garbage
import gc
gc.collect()

print(" Cleanup completed")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Summary and Next Steps

# COMMAND ----------

print(" DATA PROCESSING SUMMARY")
print("=" * 50)
print(" Successfully read secrets from Azure Key Vault")
print(" Connected to Azure Data Lake Gen2")
print(" Processed and analyzed employee data")
print(" Converted timestamps to datetime format")
print(" Flattened JSON labour_detail column")
print(" Performed advanced data transformations")
print(" Created multiple derived datasets")
print(" Written data back to Data Lake Gen2 in multiple formats")
print(" Created external tables for SQL access")
print(" Validated data integrity")

print("\n DATASETS CREATED:")
print("- employee_processed/ (partitioned by department, year, month)")
print("- employee_summary/ (business-ready format)")
print("- department_summary/ (aggregated department metrics)")
print("- skills_analysis/ (skills-based analysis)")

print("\n NEXT STEPS:")
print("1. Set up automated data quality monitoring")
print("2. Implement incremental data processing")
print("3. Create data visualization dashboards")
print("4. Set up alerts for data anomalies")
print("5. Implement data lineage tracking")

print("\n OPTIMIZATION RECOMMENDATIONS:")
print("- Consider using Delta Lake for better performance")
print("- Implement Z-ordering for frequently queried columns")
print("- Set up automatic table optimization")
print("- Monitor partition sizes and adjust partitioning strategy")

# COMMAND ----------

# Stop Spark session
# spark.stop()